# P4 — интерфейс и сдача

Этот ноутбук фиксирует, что именно было сделано в части P4: Streamlit-интерфейс, визуализация маски талька, таблицы метрик, экспертная проверка, отчёты и материалы для сдачи.

Проектная роль P4: собрать результат P1/P2/P3 в понятный локальный интерфейс, который можно показать жюри даже до финальной готовности ядра. Поэтому UI работает через стабильный контракт `core.analyze()` и умеет запускаться в режиме заглушки `ANALYZE_STUB=1`.


## Что сделано

1. Переписан `app/main.py` в полноценный Streamlit-экран для демо.
2. Добавлен кэшированный вызов `analyze()`, чтобы зум и экспертные правки не запускали повторный анализ.
3. Добавлены вкладки `Маска`, `Оригинал`, `Сравнение` и слайдеры зума.
4. Добавлена таблица метрик: сорт, процент талька, уверенность классификатора, источник вердикта, время обработки, согласованность.
5. Добавлен блок экспертной проверки: подтвердить вердикт или исправить сорт и оставить комментарий.
6. Добавлены выгрузки: PDF-отчёт по снимку, JSON результата, JSON экспертной правки и CSV по пачке изображений.
7. Обновлены `README.md` и `notebooks/README.md` под текущий скоуп: только тальк + классификатор сорта.
8. Добавлены сдачные материалы в `docs/`: 4 ссылки, сценарий демо-видео, структура презентации.


In [ ]:
from pathlib import Path
import sys

REPO = Path.cwd()
if not (REPO / 'app').exists():
    REPO = Path.cwd().parent
sys.path.insert(0, str(REPO))

print('Repo:', REPO)
print('app/main.py:', (REPO / 'app' / 'main.py').exists())
print('docs/submission.md:', (REPO / 'docs' / 'submission.md').exists())


## Контракт, на котором держится интерфейс

P4 не должен угадывать внутренности P1/P2. Он берёт готовый словарь из `core.analyze()`:

- `verdict` — сорт руды от классификатора или fallback;
- `talc_pct` — процент талька от сегментации;
- `talc_mask_png_b64` — PNG-оверлей маски талька;
- `consistency_check` — текстовая проверка согласованности сорта с процентом талька;
- `classifier_confidence` — уверенность P2, когда веса доступны;
- `processing_time_sec` — время обработки;
- `verdict_source` — источник вердикта: classifier/fallback/stub.


In [ ]:
import io
import os
import numpy as np
from PIL import Image

# Для демонстрации P4 ядро можно форсировать в режим заглушки.
os.environ['ANALYZE_STUB'] = '1'

from core import analyze as analyze_mod
from core import report

# Синтетическая картинка нужна только чтобы показать контракт без реального датасета.
img = np.full((240, 320, 3), 185, dtype=np.uint8)
img[70:170, 110:230] = [70, 70, 85]
buf = io.BytesIO()
Image.fromarray(img).save(buf, format='PNG')

result = analyze_mod.analyze(buf.getvalue(), image_id='demo_synthetic.png')
result


## Как это отображается в UI

`app/main.py` превращает результат выше в несколько рабочих зон:

- верхний вердикт с цветовым статусом;
- быстрые метрики `talc_pct`, уверенность, источник и время;
- визуальный блок с маской/оригиналом/сравнением;
- таблицу всех полей для геолога;
- экспертную проверку и кнопки скачивания.


In [ ]:
try:
    import pandas as pd
    display(pd.DataFrame([{
        'Файл': result['image_id'],
        'Сорт': result['verdict'],
        'Тальк, %': result['talc_pct'],
        'Источник': result.get('verdict_source'),
        'Согласованность': result.get('consistency_check'),
    }]))
except Exception:
    print(result)


## Проверка CSV-выгрузки

В интерфейсе CSV скачивается для всей пачки изображений. Здесь показан тот же вызов на одном тестовом результате.


In [ ]:
print(report.results_to_csv([result]))


## Проверка наличия сдачных материалов

Эти файлы нужны не для модели, а для финальной упаковки проекта: ссылка на репозиторий, облако, презентацию и деплой; сценарий видео; план презентации.


In [ ]:
for path in [
    REPO / 'docs' / 'submission.md',
    REPO / 'docs' / 'demo_video_script.md',
    REPO / 'docs' / 'presentation_outline.md',
    REPO / 'README.md',
    REPO / 'app' / 'main.py',
]:
    print(f'{path.relative_to(REPO)}:', 'OK' if path.exists() else 'missing')


## Как запустить локально

```bash
pip install -r requirements.txt
streamlit run app/main.py
```

Если нужно проверить только интерфейс без готовых весов/сегментации:

```bash
ANALYZE_STUB=1 streamlit run app/main.py
```

После загрузки изображения в UI должны быть видны: вердикт, маска/оригинал/сравнение, таблица метрик, экспертная проверка и кнопки скачивания отчётов.
